# 10 · Post-training: LoRA (stage S9)

Two LoRA adapters on the attention projections of the core layers teach the model to answer from fewer visual tokens.
Each learns from the teacher cache of nb09 (the full-token model): variant A pairs every student with the teacher at the
same loop count, variant B pairs compressed students with a fixed deeper teacher. This notebook checks the code only;
training runs only behind `RUN_SMOKE`. Settings: `configs/train_lora.yaml`.

**How to run:** Run All, no edits. `RUN_SMOKE` and `RUN_EVAL_PROBE` are off. Needs the S8 teacher cache (nb09 step 6)
and one free GPU pair among 0-3. Paste the outputs.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import socket

# GPU choice must precede CUDA init, which importing adaloop.api already does: the one import outside adaloop.api
from utils.gpus import select_gpu_group

GPU_IDS = select_gpu_group("configs/mvp.yaml")  # first free pair of gpu_groups (0-3 only)

In [ ]:
from adaloop import api

ROOT, cfg = api.setup_notebook("configs/train_lora.yaml")
print("ROOT", ROOT, "| config_hash", api.config_hash(cfg), "| git", api.git_hash())
assert cfg["compressor"] != "none", "compressor must not be none (S5 froze divprune in configs/mvp.yaml)"

In [ ]:
import torch

ON_GPU_HOST = api.is_gpu_host()  # Inara or PACE
print("host", socket.gethostname(), "| on Inara or PACE:", ON_GPU_HOST)
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
for i in range(torch.cuda.device_count()):
    total_gib = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i} | {torch.cuda.get_device_name(i)} | {total_gib:.0f} GiB")

In [ ]:
import pandas as pd

In [ ]:
# Settings (values live in configs/train_lora.yaml)
MODEL_ID = cfg["model_id"]
N_SMOKE = cfg["smoke"]["n_per_benchmark"]  # lora_train items per benchmark
N_TARGETS = 48  # q/k/v/o of the 12 core layers (checked against the config regex on attach)
R_MAX = max(cfg["teacher"]["R_list"])
RUN_SMOKE = False  # True: 100 training steps per variant (Step-7)
RUN_EVAL_PROBE = False  # True: needs the smoke adapters (Step-8)
print("model", MODEL_ID, "| smoke items per benchmark", N_SMOKE, "| R_max", R_MAX)

### Step-1:- Load the model and the data

The teacher records of the smoke items, one `TeacherTarget` per (item, R). Variant A and B come from the same
config with a different `teacher_R` rule.

In [ ]:
print("[STAGE] load model")
vlm = api.load_models(cfg, 1)[0]
print("loaded", type(vlm.model).__name__)

In [ ]:
items = api.teacher_items(cfg, ROOT, limit=N_SMOKE)
targets = api.load_targets(items, cfg, ROOT)
vcfg = api.variant_config(cfg, "A")
print(f"{len(items)} items | {len(targets)} teacher records | variant {api.variant_name(vcfg)}")

### Step-2:- One optimizer step on a throwaway adapter

One step on four items at R = 3, k = 0.25, then the adapter is removed again. Only LoRA weights may train and the base
model must not change; the step touches the 48 LoRA modules.

In [ ]:
print("[STAGE] one optimizer step")
r = api.one_step_check(vlm, items[:4], targets, vcfg, R=3, k=0.25)
r

In [ ]:
assert r["n_trainable_not_lora"] == 0, r
assert r["n_trainable_tensors"] == 2 * N_TARGETS, r
assert r["n_base_changed"] == 0, r
assert r["n_lora_modules_changed"] == N_TARGETS, r
print(f"✓ one step: {r['n_trainable_tensors']} trainable tensors, all LoRA; base changed {r['n_base_changed']}; "
      f"{r['n_lora_modules_changed']} LoRA modules changed")

### Step-3:- Attach the adapter

The adapter adds A and B matrices to the 48 attention projections with B at zero, so the model's output is unchanged
at the start. The regex must hit exactly those modules.

In [ ]:
pm = api.attach_lora(vlm, vcfg)
names = api.lora_target_names(vlm, vcfg)
names

In [ ]:
assert len(names) == api.expected_target_count(vlm) == N_TARGETS, (len(names), api.expected_target_count(vlm))
assert all("lora_" in name for name in api.lora_parameters(vlm)), list(api.lora_parameters(vlm))
print(f"✓ adapter attached: {len(names)} target modules, every trainable tensor is a LoRA weight")

### Step-4:- Gradients at R = 1, 3, 5

One backward pass per loop count at k = 0.25. Every LoRA module must get a gradient at every R; a module without one
would mean the recurrence cuts it off from the loss.

In [ ]:
print("[STAGE] gradients")
grad_rows = {}
for R in (1, 3, 5):
    grad_rows[R] = pd.DataFrame(api.lora_grad_norms(vlm, items[0], targets, vcfg, R, 0.25))
grads = pd.concat(grad_rows.values())
grads.groupby("R").agg(modules=("module", "count"), layers=("layer", "nunique"),
                       min_grad=("grad_norm", "min"), max_grad=("grad_norm", "max"))

In [ ]:
for R, rows in grad_rows.items():
    assert len(rows) == N_TARGETS, (R, len(rows))
    assert rows.layer.nunique() == N_TARGETS // 4, (R, rows.layer.nunique())
    assert rows.grad_norm.min() > 0, (R, rows.sort_values("grad_norm").head(3))
print(f"✓ gradients reach all {N_TARGETS} LoRA modules (12 layers) at R = 1, 3, 5")

### Step-5:- Loop count of the student

The student must run each core layer R times for the sampled R. The total is 12 + 12R layer calls (6 + 12R + 6 with the
outer layers).

In [ ]:
print("[STAGE] layer calls")
calls = {}
for R in (1, 3, 5):
    calls[R] = api.student_layer_calls(vlm, items[0], targets, vcfg, R, 0.25)
    assert calls[R] == api.expected_layer_calls(vlm) and sum(calls[R]) == 12 + 12 * R, (R, calls[R])
calls

In [ ]:
print("✓ layer calls match the loop count: sum = 12 + 12R for R = 1, 3, 5")

### Step-6:- Teacher lookup

Which teacher record a sample is paired with. The rule decides only for compressed samples (k < 1); full-token samples
always pair with the student's own R. Variant A uses `same`; variant B uses the rule in `cfg["lora_variants"]`.

In [ ]:
pairs = pd.DataFrame(api.pairing_table(cfg, ["same", "fixed:5", "fixed:3", "plus:1"]))
pairs.pivot_table(index=["rule", "k"], columns="R", values="R_teacher")

In [ ]:
low = pairs[pairs.k < 1.0]
full = pairs[pairs.k == 1.0]
assert (full.R_teacher == full.R).all(), "k = 1 must pair with the student's own R (every rule)"
same = low[low.rule == "same"]
assert (same.R_teacher == same.R).all(), "same: teacher R = student R"
fixed5 = low[low.rule == "fixed:5"]
assert (fixed5.R_teacher == 5).all(), "fixed:5: teacher R = 5"
fixed3 = low[low.rule == "fixed:3"]
assert (fixed3.R_teacher == 3).all(), "fixed:3: teacher R = 3"
plus1 = low[low.rule == "plus:1"]
assert (plus1.R_teacher == (plus1.R + 1).clip(upper=R_MAX)).all(), "plus:1: teacher R = min(R + 1, R_max)"
print("✓ pairing rules: same, fixed:N, plus:N with min(R + N, R_max); k = 1 always same R")

In [ ]:
print("[STAGE] schedule vs teacher records")
schedules = {}
for key in ("A", "B"):
    vc = api.variant_config(cfg, key)
    t = pd.DataFrame(api.schedule_table(items, targets, vc, 200))
    expected = [api.teacher_depth(cfg["lora_variants"][key], row.R, row.k) for row in t.itertuples()]
    assert (t.R_teacher == expected).all(), f"variant {key}: pairing differs from teacher_depth"
    assert (t.record_R == t.R_teacher).all(), f"variant {key}: record R is not the paired teacher R"
    assert (t.record_item_id == t.item_id).all(), f"variant {key}: record belongs to another item"
    schedules[key] = t
a, b = schedules["A"], schedules["B"]
assert a[["benchmark", "item_id", "k", "R"]].equals(b[["benchmark", "item_id", "k", "R"]]), "A and B draw other samples"
print(f"A rule {cfg['lora_variants']['A']} | B rule {cfg['lora_variants']['B']}")
b.head(8)

In [ ]:
print("✓ schedule: records paired as the rule says (A and B), same samples in A and B")

### Step-7:- Loss at k = 1

With k = 1 the student is the teacher (adapter at zero, same R), so the loss should be zero. It is not exactly zero:
the cache comes from batched decoding with the KV cache, the student from one full forward, so BF16 noise remains
(nb09 step 3 measures the same kind of gap on the cache). The threshold is for that noise only.

In [ ]:
print("[STAGE] loss at k = 1")
probe = pd.DataFrame(api.probe_loss(vlm, items[:cfg["smoke"]["probe_items"]], targets, vcfg, ks=[1.0]))
probe.describe()

In [ ]:
assert probe.loss.max() < 1e-2, probe.sort_values("loss", ascending=False).head(3)
print(f"✓ loss at k = 1: max {probe.loss.max():.2e} over {len(probe)} samples (BF16 noise)")

### Step-8:- Smoke training (behind `RUN_SMOKE`)

For each variant: probe loss, a short run, probe loss again, and the step time for the full-run estimate. A run folder
that already has `train_state.pt` is resumed, so its "before" loss is not an untrained one; the fall is asserted only
for a fresh run.

In [ ]:
api.remove_lora(pm)  # the adapter above is done; smoke runs attach their own
probe_items = items[:cfg["smoke"]["probe_items"]]
if RUN_SMOKE:
    for key in ("A", "B"):
        print(f"[STAGE] smoke training, variant {key}")
        vc = api.variant_config(cfg, key)
        run_dir = api.lora_run_dir(vc, ROOT, "smoke")
        fresh = not (run_dir / "train_state.pt").exists()
        pm = api.attach_lora(vlm, vc)
        before = pd.DataFrame(api.probe_loss(vlm, probe_items, targets, vc))
        history = api.train_lora(vlm, pm, items, targets, vc, run_dir, steps=cfg["smoke"]["steps"],
                                 eff_batch=cfg["smoke"]["eff_batch"])
        after = pd.DataFrame(api.probe_loss(vlm, probe_items, targets, vc))
        if fresh:
            assert after.loss.mean() < before.loss.mean(), (key, before.loss.mean(), after.loss.mean())
        print(f"{key}: probe loss {before.loss.mean():.4f} -> {after.loss.mean():.4f} | fresh run: {fresh}")
        api.remove_lora(pm)
        print(api.lora_run_estimate(history, vc, cfg["smoke"]["eff_batch"]))
else:
    print("skipped: RUN_SMOKE is False")

### Step-9:- Eval probe (behind `RUN_EVAL_PROBE`)

Greedy answers on dev items at k in {0.25, 1.0} and R in {1, 3, 5}, with the adapter off (base) and with the saved smoke
adapter. A probe to see the direction, not a result. Dev only: the test split is frozen until S12.

In [ ]:
if RUN_EVAL_PROBE:
    print("[STAGE] eval probe")
    n_dev = cfg["smoke"]["eval_items_per_benchmark"]
    dev = api.load_gqa(cfg, "dev", limit=n_dev, root=ROOT) + api.load_textvqa(cfg, "dev", limit=n_dev, root=ROOT)
    rows = []
    for key in ("A", "B"):
        vc = api.variant_config(cfg, key)
        pm = api.load_lora(vlm, api.lora_run_dir(vc, ROOT, "smoke") / "adapter")
        rows += api.lora_eval_probe(vlm, pm, dev, vc)
        api.remove_lora(pm)
    table = pd.DataFrame(rows).groupby(["model_variant", "k", "R"]).correct.mean().unstack("R").round(3)
    print(table.to_string())
else:
    print("skipped: RUN_EVAL_PROBE is False")